<a href="https://colab.research.google.com/github/manojrd21/ai-appointment-bot/blob/main/appointment_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate dateparser -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.3 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline
import dateparser
from datetime import datetime

# Load google/flan-t5-large
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    framework="pt"  # Force PyTorch instead of TensorFlow
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
def extract_datetime(user_input):
    prompt = (
        f"Your task is to extract the correct day and time from this sentence:\n"
        f"\"{user_input}\"\n\n"
        f"Respond only with the exact date and time mentioned by the user, like: 'this Saturday evening' or 'July 2nd at 11 AM'. "
        f"Don't make up anything or change the meaning."
    )

    print("Running model...")
    result = generator(prompt, max_new_tokens=50)[0]['generated_text']
    return result.strip()

In [ ]:
!pip install parsedatetime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.9 MB/s eta 0:00:00


In [ ]:
import parsedatetime as pdt
from datetime import datetime

def parse_to_date_and_time(extracted_text):
    cal = pdt.Calendar()

    # Parse using current datetime as reference
    parsed_time, status = cal.parseDT(extracted_text, sourceTime=datetime.now())

    if status:
        return parsed_time.strftime("Date: %Y-%m-%d, Time: %I:%M %p")
    else:
        return f"Could not parse: {extracted_text}"

In [ ]:
def handle_booking_request(user_input):
    extracted = extract_datetime(user_input)
    structured = parse_to_date_and_time(extracted)

    if "Could not parse" in structured:
        return "Sorry, I couldn't understand the date/time you mentioned."

    return f"""I understood your preferred time as: {extracted}
Parsed as: {structured}
Please complete your booking here:
https://cal.com/manoj-dhanawade/counselor-session"""

# Example
print(handle_booking_request("Can I talk to a counselor on Monday at 4 PM?"))


Running model...
I understood your preferred time as: Monday at 4 PM
Parsed as: Date: 2025-06-23, Time: 04:00 PM
Please complete your booking here:
https://cal.com/manoj-dhanawade/counselor-session
